# Etapa 4 — Avaliação e seleção de modelos

Continuação do `03_treinamento_modelos.ipynb`. Aqui o modelo final é escolhido e, só então, o conjunto de teste é usado pela primeira vez.

Roteiro:
1. Configuração, dados e modelos ajustados
2. Seleção do modelo por validação cruzada aninhada, e do limiar de decisão (sem olhar o teste)
3. Avaliação no conjunto de teste
4. Robustez: o resultado depende da divisão treino/teste?
5. Interpretação do modelo escolhido
6. Conclusões

A Aula 05 separa dois processos: **seleção** (escolher entre modelos e níveis de flexibilidade) e **avaliação** (estimar o erro do modelo escolhido em dados novos). Se o teste fosse usado para escolher o modelo, deixaria de ser uma estimativa honesta. A lógica está em `src/evaluate.py`.

Este notebook leva cerca de 6 minutos para rodar.

## 1. Configuração, dados e modelos ajustados

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src import utils, preprocessing, features, train, evaluate

utils.setup_plotting()

Os dados, a divisão treino/teste e o ajuste dos modelos são refeitos com as mesmas funções e a mesma semente dos notebooks 02 e 03, então os resultados são idênticos.

In [ ]:
patients, drugs, data_dictionary = preprocessing.load_data()
variable_groups = preprocessing.get_variable_groups(patients)
target = preprocessing.create_target(patients)
target.index = patients["inpatient.number"].values

X, _ = features.build_model_matrix(patients, drugs, variable_groups)
X_train, X_test, y_train, y_test = preprocessing.split_train_test(X, target)

selected_features = features.load_selected_features()
X_train_selected, X_test_selected = X_train[selected_features], X_test[selected_features]

cv_summary, searches = train.tune_models(X_train_selected, y_train)
print(f"\nTreino: {len(y_train)} internações | Teste: {len(y_test)} internações ({int(y_test.sum())} eventos) | {len(selected_features)} atributos")

## 2. Seleção do modelo

**Por que não usar a validação cruzada da etapa 3.** Naquela validação os 23 atributos já estavam fixos, e tinham sido escolhidos com todo o treino, inclusive as internações que serviam de validação. Isso tem dois efeitos: a AUC sai otimista, e a comparação favorece os modelos lineares, porque os atributos foram escolhidos por um modelo linear (o Lasso).

**Validação cruzada aninhada.** Os mesmos 10 *folds* da etapa 3 formam o laço externo. Para cada *fold* externo:
1. toda a seleção de atributos da etapa 2 é refeita com os outros nove *folds*;
2. cada um dos seis modelos tem os hiperparâmetros ajustados por validação cruzada dentro desses nove *folds* (laço interno);
3. a AUC de cada modelo é medida no *fold* deixado de fora, que não participou de nenhuma decisão.

**Como ler a tabela**: `nested_auc_mean` e `nested_auc_se` são a média e o erro padrão da AUC nos 10 *folds* externos (a estimativa honesta); `step3_cv_auc` é a AUC da validação cruzada da etapa 3; `optimism` é a diferença entre as duas, ou seja, quanto da AUC da etapa 3 vinha de escolher os atributos olhando os dados de validação.

In [ ]:
outer_scores, nested_n_features = evaluate.nested_cv_model_comparison(X_train, y_train)
nested_summary = evaluate.summarize_nested_cv(outer_scores, cv_summary["cv_auc_mean"])

nested_summary.to_csv(utils.RESULTS_DIR / "model_nested_cv_results.csv")
print(f"Atributos selecionados em cada fold externo: de {nested_n_features.min()} a {nested_n_features.max()} (mediana {nested_n_features.median():.0f})")
nested_summary

**Critério de escolha**: maior AUC na validação aninhada; entre os modelos a até um erro padrão do melhor, fica o mais simples (mesma lógica da regra de um erro padrão da Aula 07). A ordem de simplicidade põe primeiro os modelos lineares, cujos coeficientes são diretamente interpretáveis.

In [ ]:
# Simplest first: linear models (interpretable coefficients), then non-linear ones
SIMPLICITY_ORDER = ["Regressão logística (Ridge)", "LDA", "KNN", "SVM (kernel RBF)", "Random Forest", "Gradient Boosting"]

best = nested_summary.iloc[0]
within_one_se = nested_summary.index[nested_summary["nested_auc_mean"] >= best["nested_auc_mean"] - best["nested_auc_se"]].tolist()
CHOSEN_MODEL = next(model for model in SIMPLICITY_ORDER if model in within_one_se)

print(f"Maior AUC aninhada: {nested_summary.index[0]} ({best['nested_auc_mean']:.3f} ± {best['nested_auc_se']:.3f})")
print(f"Modelos a até um erro padrão do melhor: {len(within_one_se)} de {len(nested_summary)}")
print(f"Modelo escolhido: {CHOSEN_MODEL}")

**Comparação par a par com o modelo escolhido.** Como os *folds* externos são os mesmos para todos os modelos, a diferença pode ser calculada *fold* a *fold*, o que é mais sensível do que comparar as médias.

**Como ler a tabela**: `auc_difference_mean` é a AUC média do modelo menos a do modelo escolhido (negativo = pior que o escolhido) e `auc_difference_se` o erro padrão dessa diferença; uma diferença menor que cerca de dois erros padrão não é distinguível de ruído. `folds_better_than_reference` conta em quantos dos 10 *folds* o modelo superou o escolhido.

In [ ]:
evaluate.paired_differences(outer_scores, CHOSEN_MODEL)

### 2.1 Limiar de decisão

**Objetivo**: o modelo devolve uma probabilidade de evento para cada paciente; o limiar é o ponto de corte acima do qual o paciente é classificado como "evento". O padrão é 0,5, mas com sinal fraco as probabilidades previstas ficam concentradas perto da taxa de eventos da base (41,5%) e poucas passam de 0,5, de modo que o modelo deixaria de identificar a maioria dos pacientes com evento.

**Como é feito**: para cada internação do treino obtemos a probabilidade prevista quando ela está no *fold* de validação (ou seja, por um modelo que não a viu). Sobre essas probabilidades escolhemos o limiar que maximiza o índice de Youden (sensibilidade + especificidade − 1), que dá o mesmo peso aos dois tipos de erro. O teste não participa da escolha.

Mudar o limiar não melhora a capacidade de discriminação do modelo (a AUC é a mesma): apenas troca falsos negativos por falsos positivos.

In [ ]:
final_model = searches[CHOSEN_MODEL].best_estimator_
CHOSEN_THRESHOLD = evaluate.choose_threshold_cv(final_model, X_train_selected, y_train)

print(f"Limiar escolhido pela validação cruzada do treino: {CHOSEN_THRESHOLD:.3f} (taxa de eventos no treino: {y_train.mean():.3f})")

## 3. Avaliação no conjunto de teste

**Objetivo**: estimar o desempenho em pacientes que não participaram de nenhuma decisão (seleção de atributos, imputação, hiperparâmetros, escolha do modelo e do limiar). Os seis modelos são avaliados para dar contexto, mas a escolha já foi feita na seção 2.

**Como ler a tabela**: `test_auc` é a AUC no teste; `auc_ci_low` e `auc_ci_high` são o intervalo de confiança de 95% por *bootstrap* (2000 reamostras do teste, Aula 05); `nested_auc_mean` repete a AUC da validação aninhada da seção 2, para comparação.

In [ ]:
test_results = evaluate.evaluate_on_test(searches, X_test_selected, y_test)
test_results = test_results.join(nested_summary[["nested_auc_mean"]]).sort_values("nested_auc_mean", ascending=False)

test_results.to_csv(utils.RESULTS_DIR / "model_test_results.csv")
test_results

**Como ler o gráfico**: curva ROC no conjunto de teste. Cada ponto da curva é um limiar de decisão; o eixo vertical é a sensibilidade e o horizontal a taxa de falsos positivos. A diagonal pontilhada é um classificador ao acaso (AUC 0,5); quanto mais a curva se afasta dela em direção ao canto superior esquerdo, melhor. Em azul o modelo escolhido, em cinza os demais.

In [ ]:
fig = evaluate.plot_roc_curves(searches, X_test_selected, y_test, highlight=CHOSEN_MODEL)
utils.save_fig(fig, "roc_curves_test")

**Modelo escolhido no teste, com o limiar padrão e com o limiar escolhido na seção 2.1.**

**Como ler a tabela**: `sensitivity` é a fração dos pacientes com evento que o modelo identifica; `specificity` a fração dos sem evento classificados corretamente; `balanced_accuracy` a média das duas; `accuracy` a fração de acertos (referência: classificar todos como "sem evento" acerta 58,5%); `positive_predictive_value` é a fração dos classificados como "evento" que de fato tiveram evento, e `negative_predictive_value` o análogo para "sem evento".

In [ ]:
threshold_results = evaluate.threshold_metrics(
    final_model, X_test_selected, y_test,
    {"Limiar padrão (0,5)": 0.5, "Limiar escolhido na validação cruzada": CHOSEN_THRESHOLD})

threshold_results.to_csv(utils.RESULTS_DIR / "final_model_threshold_metrics.csv", index_label="threshold_rule")
threshold_results

**Matriz de confusão** do modelo escolhido no teste, com o limiar escolhido. Linhas são o desfecho real e colunas a previsão.

In [ ]:
evaluate.confusion_table(final_model, X_test_selected, y_test, threshold=CHOSEN_THRESHOLD)

## 4. Robustez: o resultado depende da divisão treino/teste?

**Objetivo**: a AUC no teste (seção 3) ficou abaixo da estimada pela validação aninhada (seção 2). Um teste de 402 internações varia bastante conforme quais pacientes caem nele. Esta seção mede essa variação e estima o desempenho **esperado do método**, independentemente de uma divisão específica.

**Como é feito**: o procedimento completo é repetido em 50 divisões 80/20 estratificadas diferentes. Em cada uma:
1. novo sorteio de treino e teste;
2. toda a seleção de atributos da etapa 2 é refeita **só com aquele treino** (filtro, Lasso com a regra de um erro padrão, frequência de seleção >= 50%);
3. o modelo escolhido é ajustado naquele treino (mesma grade e validação cruzada da etapa 3);
4. a AUC é medida naquele teste.

Nenhuma rodada usa informação do seu próprio teste. A semente 42 reproduz exatamente a divisão usada no trabalho, o que serve de conferência. O conjunto de teste oficial e o modelo final não mudam: esta análise é só um complemento, e **não** é usada para escolher uma divisão mais favorável.

A célula abaixo leva alguns minutos.

In [ ]:
N_SPLITS_ROBUSTNESS = 50

split_results, feature_share = evaluate.repeated_split_assessment(
    X, target, CHOSEN_MODEL, seeds=range(N_SPLITS_ROBUSTNESS))

split_results.to_csv(utils.RESULTS_DIR / "repeated_splits_auc.csv")
feature_share.to_csv(utils.RESULTS_DIR / "repeated_splits_feature_share.csv", index_label="feature")

official_auc = test_results.loc[CHOSEN_MODEL, "test_auc"]
assert abs(split_results.loc[utils.RANDOM_STATE, "test_auc"] - official_auc) < 0.001  # seed 42 = official split

**Como ler a tabela**: resumo das 50 rodadas. `test_auc` é a AUC no teste de cada rodada (a estimativa honesta); `cv_auc` é a AUC em validação cruzada dentro do treino daquela rodada (otimista, como na etapa 3); `n_features` é o número de atributos selecionados. A diferença entre as médias de `cv_auc` e `test_auc` mede o tamanho do otimismo.

In [ ]:
summary = split_results[["test_auc", "cv_auc", "n_features"]].agg(["mean", "std", "min", "median", "max"]).round(3)

share_below = (split_results["test_auc"] <= official_auc).mean()
print(f"AUC no teste oficial: {official_auc:.3f}")
print(f"AUC média nas {N_SPLITS_ROBUSTNESS} divisões: {split_results['test_auc'].mean():.3f} (desvio padrão {split_results['test_auc'].std():.3f})")
print(f"Divisões com AUC igual ou menor que a oficial: {share_below:.0%}")
summary

**Como ler o gráfico**: histograma da AUC no teste nas 50 divisões. A linha tracejada é a média (desempenho esperado do método) e a linha azul é a divisão usada no trabalho. Quanto mais à esquerda a linha azul, mais desfavorável foi o sorteio.

In [ ]:
fig = evaluate.plot_repeated_split_auc(split_results["test_auc"], official_auc)
utils.save_fig(fig, "repeated_splits_auc")

**Estabilidade dos atributos entre divisões**. `share_of_splits` é a fração das 50 rodadas em que o atributo entrou no conjunto final; `in_final_set` indica se ele está nos 23 atributos do trabalho. Atributos com fração alta são escolhidos qualquer que seja o treino.

In [ ]:
feature_share_table = feature_share.to_frame()
feature_share_table["in_final_set"] = feature_share_table.index.isin(selected_features)
print(f"Atributos escolhidos em pelo menos metade das divisões: {(feature_share >= 0.5).sum()}")
feature_share_table.head(30)

## 5. Interpretação do modelo escolhido

**Como ler a tabela**: `coefficient` é o coeficiente na escala padronizada; `odds_ratio_per_sd` é por quanto a chance de óbito ou readmissão é multiplicada quando o atributo aumenta um desvio padrão, mantidos os demais (Aula 04). Acima de 1 aumenta o risco, abaixo de 1 reduz. Os coeficientes são encolhidos pela penalidade L2 e descrevem associação, não causa.

In [ ]:
odds_ratios = evaluate.logistic_odds_ratios(final_model, selected_features)
odds_ratios.to_csv(utils.RESULTS_DIR / "final_model_odds_ratios.csv", index_label="feature")
odds_ratios

## 6. Conclusões

A tabela abaixo reúne os números em que as conclusões se apoiam, todos calculados neste notebook ou lidos dos resultados do notebook 02.

In [ ]:
feature_sets = pd.read_csv(utils.RESULTS_DIR / "feature_set_comparison.csv", index_col=[0, 1])["auc_mean"]
chosen_threshold_row = threshold_results.loc["Limiar escolhido na validação cruzada"]

key_numbers = pd.Series({
    "AUC, validação cruzada simples (etapa 3, otimista)": cv_summary.loc[CHOSEN_MODEL, "cv_auc_mean"],
    "AUC, validação cruzada aninhada": nested_summary.loc[CHOSEN_MODEL, "nested_auc_mean"],
    "AUC, média de 50 divisões treino/teste": split_results["test_auc"].mean(),
    "AUC, desvio padrão entre as 50 divisões": split_results["test_auc"].std(),
    "AUC, conjunto de teste": official_auc,
    "AUC no teste, limite inferior do IC 95%": test_results.loc[CHOSEN_MODEL, "auc_ci_low"],
    "AUC no teste, limite superior do IC 95%": test_results.loc[CHOSEN_MODEL, "auc_ci_high"],
    "AUC aninhada, pior dos seis modelos": nested_summary["nested_auc_mean"].min(),
    "AUC aninhada, melhor dos seis modelos": nested_summary["nested_auc_mean"].max(),
    "Sensibilidade no teste (limiar escolhido)": chosen_threshold_row["sensitivity"],
    "Especificidade no teste (limiar escolhido)": chosen_threshold_row["specificity"],
    "Logística sem penalidade, todos os atributos": feature_sets[("Logística sem penalidade", "todas")],
    "Logística sem penalidade, atributos do Lasso": feature_sets[("Logística sem penalidade", "Lasso")],
    "Logística sem penalidade, atributos do filtro F": feature_sets[("Logística sem penalidade", "filtro F")],
    "Random Forest, todos os atributos": feature_sets[("Random Forest", "todas")],
    "Random Forest, atributos do Lasso": feature_sets[("Random Forest", "Lasso")],
    "Random Forest, atributos do filtro F": feature_sets[("Random Forest", "filtro F")],
}, name="valor").round(3)
key_numbers.to_frame()

### Resultado principal

Com as informações disponíveis na internação, a capacidade de prever óbito ou readmissão em 6 meses nesta base é **modesta**: AUC de cerca de 0,65 a 0,67 nas estimativas que não dependem de uma única divisão (validação aninhada e 50 divisões repetidas) e 0,60 no conjunto de teste (IC 95% de 0,54 a 0,65). Nenhuma escolha de modelo ou de técnica de seleção muda esse patamar: o limite parece estar no sinal presente nos dados, não no método.

### Técnicas de seleção de atributos

- **Filtro sem olhar o alvo** (constantes, quase constantes e pares com Spearman > 0,9): tirou 28 das 196 colunas sem custo de desempenho. Vale sempre fazer.
- **A seleção supervisionada ajuda o modelo linear e é indiferente para o não linear.** A regressão logística sem penalidade passa de 0,64 com todos os atributos para 0,66 a 0,67 com os selecionados; o Random Forest fica em 0,66 a 0,67 com ou sem seleção. Faz sentido: a floresta já ignora atributos inúteis por construção, e a regressão sem penalidade não.
- **O método embutido (Lasso) não superou um filtro univariado simples.** Com o mesmo número de atributos, o *ranking* pelo teste F deu resultado igual ou ligeiramente melhor, dentro de um erro padrão. Nesta base importa reduzir a dimensão, e não qual técnica faz a redução. A vantagem prática do Lasso é outra: ele escolhe sozinho quantos atributos manter (pela validação cruzada com a regra de um erro padrão) e evita ficar com variáveis redundantes entre si; o filtro precisa que o número seja dado de fora.
- **A seleção tem um núcleo estável e uma cauda instável.** Oito atributos são escolhidos em pelo menos 94% das 50 divisões (IC só esquerda, três indicadores de ecocardiograma não realizado, diâmetro do ventrículo esquerdo, Killip IV, ocupação agricultor e doença renal crônica). O restante varia: o número de atributos selecionados vai de 6 a 53 conforme a divisão, e 4 dos 23 atributos finais aparecem em menos da metade delas. As conclusões clínicas devem se apoiar no núcleo.
- **Critérios de informação** (AIC e BIC) concordam com a validação cruzada em rejeitar o uso de todos os atributos, mas discordam entre si sobre o tamanho ideal (o BIC prefere 17, o AIC 42). Servem como conferência, não como critério único.
- **A importância por permutação do Random Forest** discrimina pouco quando há muitos atributos correlacionados: a maior queda individual de AUC foi de 0,004. Ela aponta o mesmo núcleo e acrescenta troponina, D-dímero e albumina, que o Lasso deixou perto do corte.

### Modelos

- **Modelos lineares e não lineares têm o mesmo desempenho.** Na validação aninhada os seis ficam entre 0,656 e 0,669, com erro padrão de cerca de 0,012. Só o Gradient Boosting fica consistentemente abaixo do modelo escolhido (diferença de 0,013, pouco mais de dois erros padrão). Não há evidência de não linearidade ou interação que os modelos flexíveis consigam explorar com 1600 internações.
- **Modelo escolhido: regressão logística com penalidade L2**, por parcimônia: está a menos de um erro padrão do melhor e é o mais simples de interpretar. A penalidade ótima é forte (C ≈ 0,003), outro sinal de que os dados sustentam pouco mais que efeitos pequenos e aditivos.
- **O Random Forest é o que mais sobreajusta** (AUC 0,84 no treino contra 0,69 na validação), sem que isso se traduza em ganho.
- **No teste, com o limiar escolhido por validação cruzada (0,427)**, o modelo identifica 46% dos pacientes com evento, com especificidade de 67%; metade dos alertas é correta. Com o limiar padrão de 0,5 identificaria só 23%.

### Técnicas de validação

- **Selecionar atributos fora da validação cruzada infla o resultado.** A validação cruzada simples da etapa 3 deu 0,71; refazendo a seleção dentro de cada *fold* (validação aninhada), a mesma medida cai para 0,67. O otimismo foi de cerca de 0,04.
- **O otimismo não é igual para todos os modelos**: foi de 0,04 para logística, LDA e SVM e de 0,02 a 0,03 para KNN e Random Forest. Na validação simples os lineares pareciam melhores que os demais; na aninhada, empatam. A comparação entre modelos só é confiável quando a seleção está dentro do laço.
- **Um único conjunto de teste deste tamanho é ruidoso.** Entre 50 divisões diferentes a AUC no teste teve desvio padrão de 0,024 e variou de 0,58 a 0,70; a divisão usada neste trabalho ficou entre as 4% piores. O teste único continua sendo a avaliação mais limpa, mas deve ser lido junto com o intervalo de confiança e com a média das divisões repetidas.
- **Validação aninhada e divisões repetidas contam a mesma história** (0,67 e 0,65), por caminhos independentes, e são as estimativas a reportar como desempenho esperado.
- **O *bootstrap*** foi usado para o que a Aula 05 propõe, medir incerteza: a frequência de seleção de cada atributo e o intervalo de confiança da AUC no teste.
- **A regra de um erro padrão** foi usada duas vezes (penalidade do Lasso e escolha do modelo) e nas duas levou à opção mais simples sem perda mensurável.

### Limitações

- Base de um único hospital; o desempenho em outra população não foi testado.
- Vários atributos selecionados descrevem a conduta do hospital (exame não pedido, fármaco administrado), não o estado do paciente. Ajudam a prever, mas não indicam causa e podem não valer em outro serviço.
- A calibração das probabilidades previstas não foi avaliada, só a discriminação (AUC) e as métricas no limiar.
- A análise exploratória e a escolha da variável alvo usaram a base inteira, antes da separação do conjunto de teste.